# Experiment 2: Rank ablation with fixed LR
Runs Muon and AdamW across LoRA ranks 4, 8, 16, 32, 64.

## Cell 1: Install

In [ ]:
!pip install -q transformers peft datasets accelerate trl nbformat
!pip install --upgrade Pillow

## Cell 2: Load Muon

In [ ]:
%run /home/ubuntu/thesis-storage-1/muon.ipynb

## Cell 3: Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from torch.utils.data import DataLoader
import copy
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## Cell 4: Config

In [ ]:
MODEL_NAME  = 'microsoft/Phi-4-mini-instruct' #Replace it with your model name
LORA_RANKS  = [4, 8, 16, 32, 64]   # ranks to test
LORA_ALPHA  = 32
LORA_DROPOUT = 0.05
# We attach LoRA adapters to all attention and MLP projection layers.
# This covers query, key, value, output projections in attention, and gate, up, down projections in the feed-forward block
LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                'gate_proj', 'up_proj', 'down_proj']

MAX_STEPS   = 7989   # 3 epochs
BATCH_SIZE  = 8
MAX_SEQ_LEN = 512
GRAD_ACCUM  = 8

MUON_LR           = 0.002 #Sub-optimal LR.
MUON_MOMENTUM     = 0.95
MUON_WD           = 0.1
MUON_UPDATE_SCALE = 0.3

ADAMW_LR = 3e-4
ADAMW_WD = 0.1

SVD_TRACK_EVERY = 1000

SAVE_DIR = '/home/ubuntu/thesis-storage-1/experiment_2_results' #Replace with ur file path
os.makedirs(SAVE_DIR, exist_ok=True)

print('Config set.')

## Cell 5: Load Tokenizer and Base Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=False,
).cuda()

print('Base model loaded.')
print('Parameters:', round(sum(p.numel() for p in base_model.parameters()) / 1e9, 2), 'B')

## Cell 6: Load and Tokenize Dataset

In [ ]:
dataset = load_dataset('zwhe99/commonsense_170k', split='train')
print('Dataset size:', len(dataset))

def tokenize(example):
    text = f"### Instruction:\n{example['instruction']}\n### Response:\n{example['output']}"
    tokens = tokenizer(
        text, truncation=True, max_length=MAX_SEQ_LEN,
        padding='max_length', return_tensors='pt',
    )
    tokens['labels'] = tokens['input_ids'].clone()
    return {k: v.squeeze(0) for k, v in tokens.items()}

tokenized = dataset.map(tokenize, remove_columns=dataset.column_names)
tokenized.set_format('torch')
print('Tokenized. Batches per epoch:', len(tokenized) // BATCH_SIZE)

## Cell 7: Train Function

In [ ]:
def train_one_run(model, dataloader, optimizer_type, rank, max_steps=MAX_STEPS, grad_accum=GRAD_ACCUM):
    print(f'\n--- Rank {rank} | {optimizer_type.upper()} ---')

    if optimizer_type == 'muon':
        muon_params, adamw_params = get_muon_and_adamw_params(model)
        optimizer = [Muon(muon_params, lr=MUON_LR, momentum=MUON_MOMENTUM,
                         weight_decay=MUON_WD, update_scale=MUON_UPDATE_SCALE)]
        if len(adamw_params) > 0:
            optimizer.append(torch.optim.AdamW(adamw_params, lr=ADAMW_LR, weight_decay=ADAMW_WD))
    else:
        optimizer = [torch.optim.AdamW(model.parameters(), lr=ADAMW_LR, weight_decay=ADAMW_WD)]

    loss_history = []
    sv_history   = {}
    model.train()
    step = 0
    data_iter = iter(dataloader)

    while step < max_steps:
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            batch = next(data_iter)

        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss / grad_accum
        loss.backward()

        if (step + 1) % grad_accum == 0:
            for opt in optimizer:
                opt.step()
                opt.zero_grad()

        loss_val = loss.item() * grad_accum
        loss_history.append((step, loss_val))

        if step % SVD_TRACK_EVERY == 0:
            sv_history[step] = track_svd(model, step)
            print(f'Step {step:4d} | Loss: {loss_val:.4f}')

        step += 1

    print(f'Final loss: {loss_val:.4f}')
    return loss_history, sv_history

## Cell 8: Run All Ranks

In [ ]:
all_results = {}

for rank in LORA_RANKS:
    print(f'\n========== RANK {rank} ==========')
    rank_results = {}

    for opt_type in ['adamw', 'muon']:
        from peft import get_peft_model, LoraConfig, TaskType
        import copy

        lora_config = LoraConfig(
            r=rank,
            lora_alpha=2 * rank,
            lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGETS,
            task_type=TaskType.CAUSAL_LM,
            bias='none',
        )

        # Move base_model to CPU before copying — avoids double GPU memory
        base_model = base_model.cpu()
        torch.cuda.empty_cache()

        # Deepcopy from CPU, apply LoRA, move to GPU
        model = get_peft_model(copy.deepcopy(base_model), lora_config)
        model = model.cuda()
        model.print_trainable_parameters()

        dataloader = DataLoader(tokenized, batch_size=BATCH_SIZE, shuffle=True)

        losses, svd = train_one_run(model, dataloader, opt_type, rank)

        rank_results[opt_type] = {'losses': losses, 'svd': svd}

        torch.save(rank_results[opt_type],
                   os.path.join(SAVE_DIR, f'rank{rank}_{opt_type}_results.pt'))
        print(f'Saved: rank{rank}_{opt_type}_results.pt')

        # Free GPU memory
        del model
        torch.cuda.empty_cache()

    all_results[rank] = rank_results

## Cell 9: Plot Loss Curves by Rank

In [ ]:
import pandas as pd

fig, axes = plt.subplots(1, len(LORA_RANKS), figsize=(20, 4), sharey=False)

for i, rank in enumerate(LORA_RANKS):
    ax = axes[i]

    adamw_steps = [x[0] for x in all_results[rank]['adamw']['losses']]
    adamw_vals  = [x[1] for x in all_results[rank]['adamw']['losses']]
    muon_steps  = [x[0] for x in all_results[rank]['muon']['losses']]
    muon_vals   = [x[1] for x in all_results[rank]['muon']['losses']]

    # skip initial spike
    adamw_pairs = [(s, v) for s, v in zip(adamw_steps, adamw_vals) if s >= 200]
    muon_pairs  = [(s, v) for s, v in zip(muon_steps, muon_vals)   if s >= 200]
    adamw_steps, adamw_vals = zip(*adamw_pairs)
    muon_steps,  muon_vals  = zip(*muon_pairs)

    # smooth
    window = 200
    adamw_smooth = pd.Series(adamw_vals).rolling(window, min_periods=1).mean()
    muon_smooth  = pd.Series(muon_vals).rolling(window,  min_periods=1).mean()

    # plot raw (faint) + smoothed (bold)
    ax.plot(adamw_steps, adamw_vals,   color='blue',   alpha=0.15)
    ax.plot(muon_steps,  muon_vals,    color='orange', alpha=0.15)
    ax.plot(adamw_steps, adamw_smooth, color='blue',   alpha=0.9, linewidth=1.5, label='AdamW')
    ax.plot(muon_steps,  muon_smooth,  color='orange', alpha=0.9, linewidth=1.5, label='Muon')

    ax.set_yscale('log')
    ax.set_title(f'Rank {rank}')
    ax.set_xlabel('Step')
    if i == 0:
        ax.set_ylabel('Loss (log scale)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Loss Curves: Muon vs AdamW across LoRA Ranks', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'rank_ablation_loss.png'), dpi=150)
plt.show()